In [ ]:
!pip install -q gradio pymupdf python-docx sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.1 MB/s eta 0:00:00


In [ ]:
import os
import re
import fitz
import docx
import gradio as gr
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s+#.]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
def extract_text_from_pdf(file_path):
    text = ""
    pdf = fitz.open(file_path)
    for page in pdf:
        text += page.get_text()
    return text


def extract_text_from_docx(file_path):
    document = docx.Document(file_path)
    return "\n".join([para.text for para in document.paragraphs])


def extract_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".pdf":
        return extract_text_from_pdf(file_path)

    elif ext == ".docx":
        return extract_text_from_docx(file_path)

    elif ext == ".txt":
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

    else:
        return ""

In [ ]:
programming_languages = [
    "python", "java", "c", "c++", "javascript", "html", "css",
    "sql", "php", "ruby", "go", "kotlin", "swift"
]

skills = [
    "machine learning", "deep learning", "nlp", "natural language processing",
    "data analysis", "pandas", "numpy", "scikit-learn", "tensorflow",
    "pytorch", "fastapi", "flask", "django", "react", "node.js",
    "git", "docker", "aws", "mysql", "postgresql", "mongodb",
    "power bi", "tableau", "excel"
]

In [ ]:
def extract_items(text, items):
    found = []
    for item in items:
        pattern = r"\b" + re.escape(item) + r"\b"
        if re.search(pattern, text):
            found.append(item)
    return found


def overlap_score(resume_items, jd_items):
    if len(jd_items) == 0:
        return 0
    return len(set(resume_items) & set(jd_items)) / len(set(jd_items))

In [ ]:
def extract_experience_score(text):
    patterns = [
        r"(\d+)\+?\s*years",
        r"(\d+)\+?\s*year",
        r"(\d+)\+?\s*yrs"
    ]

    years = []

    for pattern in patterns:
        matches = re.findall(pattern, text)
        for match in matches:
            years.append(int(match))

    if not years:
        return 0.3

    max_years = max(years)

    if max_years >= 5:
        return 1.0
    elif max_years >= 3:
        return 0.8
    elif max_years >= 1:
        return 0.6
    else:
        return 0.3

In [ ]:
def rank_bulk_resumes(resume_files, job_description):
    if not resume_files:
        return pd.DataFrame({"Error": ["Please upload at least one resume."]})

    if job_description.strip() == "":
        return pd.DataFrame({"Error": ["Please enter a job description."]})

    clean_jd = clean_text(job_description)

    jd_skills = extract_items(clean_jd, skills)
    jd_languages = extract_items(clean_jd, programming_languages)

    resume_data = []

    for file in resume_files:
        file_path = file.name
        resume_text = extract_text(file_path)

        if resume_text.strip() == "":
            continue

        clean_resume = clean_text(resume_text)

        resume_skills = extract_items(clean_resume, skills)
        resume_languages = extract_items(clean_resume, programming_languages)

        skill_score = overlap_score(resume_skills, jd_skills)
        language_score = overlap_score(resume_languages, jd_languages)
        experience_score = extract_experience_score(clean_resume)

        resume_data.append({
            "Resume Name": os.path.basename(file_path),
            "Resume Text": clean_resume,
            "Matched Skills": ", ".join(set(resume_skills) & set(jd_skills)),
            "Missing Skills": ", ".join(set(jd_skills) - set(resume_skills)),
            "Matched Languages": ", ".join(set(resume_languages) & set(jd_languages)),
            "Missing Languages": ", ".join(set(jd_languages) - set(resume_languages)),
            "Skill Score": skill_score,
            "Language Score": language_score,
            "Experience Score": experience_score
        })

    if len(resume_data) == 0:
        return pd.DataFrame({"Error": ["No valid resume text found."]})

    resume_texts = [item["Resume Text"] for item in resume_data]

    tfidf = TfidfVectorizer(stop_words="english")
    tfidf_matrix = tfidf.fit_transform(resume_texts + [clean_jd])

    resume_vectors = tfidf_matrix[:-1]
    jd_vector = tfidf_matrix[-1]

    keyword_scores = cosine_similarity(resume_vectors, jd_vector).flatten()

    resume_embeddings = model.encode(resume_texts, show_progress_bar=False)
    jd_embedding = model.encode([clean_jd])

    semantic_scores = cosine_similarity(resume_embeddings, jd_embedding).flatten()

    results = []

    for i, item in enumerate(resume_data):
        skill_score = item["Skill Score"]
        language_score = item["Language Score"]
        semantic_score = semantic_scores[i]
        keyword_score = keyword_scores[i]
        experience_score = item["Experience Score"]

        final_score = (
            0.25 * skill_score +
            0.35 * language_score +
            0.20 * semantic_score +
            0.10 * keyword_score +
            0.10 * experience_score
        )

        if final_score >= 0.85:
            recommendation = "Excellent Match"
        elif final_score >= 0.70:
            recommendation = "Good Match"
        elif final_score >= 0.50:
            recommendation = "Moderate Match"
        else:
            recommendation = "Low Match"

        results.append({
            "Resume Name": item["Resume Name"],
            "Final Score (%)": round(final_score * 100, 2),
            "Skill Score (%)": round(skill_score * 100, 2),
            "Language Score (%)": round(language_score * 100, 2),
            "Semantic Score (%)": round(semantic_score * 100, 2),
            "Keyword Score (%)": round(keyword_score * 100, 2),
            "Experience Score (%)": round(experience_score * 100, 2),

            "Matched Languages": item["Matched Languages"],
            "Missing Languages": item["Missing Languages"],
            "Recommendation": recommendation
        })

    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values(by="Final Score (%)", ascending=False)
    result_df.insert(0, "Rank", range(1, len(result_df) + 1))

    return result_df

In [ ]:
custom_css = """
:root {
    --primary: #00e5ff;
    --secondary: #7c3aed;
    --accent: #22c55e;
    --dark: #020617;
    --panel: rgba(15, 23, 42, 0.78);
    --glass: rgba(255, 255, 255, 0.08);
    --border: rgba(148, 163, 184, 0.22);
    --text: #e5e7eb;
    --muted: #94a3b8;
}

.gradio-container {
    max-width: 1450px !important;
    margin: auto !important;
    min-height: 100vh !important;
    background:
        radial-gradient(circle at 15% 10%, rgba(0, 229, 255, 0.22), transparent 30%),
        radial-gradient(circle at 85% 20%, rgba(124, 58, 237, 0.24), transparent 32%),
        radial-gradient(circle at 50% 90%, rgba(34, 197, 94, 0.13), transparent 30%),
        linear-gradient(135deg, #020617 0%, #0f172a 45%, #111827 100%) !important;
    color: var(--text) !important;
    font-family: Inter, Orbitron, system-ui, sans-serif !important;
}

/* Global text */
.gradio-container * {
    color: var(--text);
}

/* Top futuristic hero */
#hero {
    position: relative;
    overflow: hidden;
    padding: 42px 44px;
    border-radius: 30px;
    background:
        linear-gradient(135deg, rgba(15,23,42,0.92), rgba(30,41,59,0.72)),
        radial-gradient(circle at top right, rgba(0,229,255,0.28), transparent 35%);
    border: 1px solid rgba(0, 229, 255, 0.28);
    box-shadow:
        0 0 35px rgba(0, 229, 255, 0.16),
        0 28px 80px rgba(0, 0, 0, 0.42);
    margin-bottom: 26px;
}

#hero::before {
    content: "";
    position: absolute;
    inset: 0;
    background-image:
        linear-gradient(rgba(0,229,255,0.07) 1px, transparent 1px),
        linear-gradient(90deg, rgba(0,229,255,0.07) 1px, transparent 1px);
    background-size: 36px 36px;
    mask-image: linear-gradient(to bottom, black, transparent);
    pointer-events: none;
}

#hero::after {
    content: "";
    position: absolute;
    width: 260px;
    height: 260px;
    right: -90px;
    top: -90px;
    background: radial-gradient(circle, rgba(0,229,255,0.45), transparent 65%);
    filter: blur(4px);
}

#badge {
    position: relative;
    display: inline-block;
    padding: 8px 16px;
    border-radius: 999px;
    background: rgba(0, 229, 255, 0.10);
    border: 1px solid rgba(0, 229, 255, 0.42);
    color: #67e8f9 !important;
    font-size: 13px;
    font-weight: 700;
    letter-spacing: 0.4px;
    margin-bottom: 14px;
    box-shadow: 0 0 18px rgba(0,229,255,0.18);
}

#hero-title {
    position: relative;
    font-size: 46px;
    font-weight: 900;
    letter-spacing: -1.5px;
    margin-bottom: 12px;
    background: linear-gradient(90deg, #ffffff, #67e8f9, #a78bfa);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

#hero-subtitle {
    position: relative;
    font-size: 17px;
    max-width: 900px;
    line-height: 1.75;
    color: #cbd5e1 !important;
}

/* Metric cards */
.metric-grid {
    position: relative;
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 16px;
    margin-top: 30px;
}

.metric-card {
    background:
        linear-gradient(135deg, rgba(255,255,255,0.09), rgba(255,255,255,0.03));
    border: 1px solid rgba(0, 229, 255, 0.22);
    border-radius: 22px;
    padding: 18px;
    backdrop-filter: blur(18px);
    box-shadow:
        inset 0 0 22px rgba(255,255,255,0.03),
        0 12px 35px rgba(0,0,0,0.24);
    transition: all 0.25s ease;
}

.metric-card:hover {
    transform: translateY(-4px);
    border-color: rgba(0, 229, 255, 0.55);
    box-shadow: 0 0 28px rgba(0,229,255,0.18);
}

.metric-value {
    font-size: 28px;
    font-weight: 900;
    color: #67e8f9 !important;
    text-shadow: 0 0 18px rgba(0,229,255,0.4);
}

.metric-label {
    font-size: 13px;
    color: #cbd5e1 !important;
    margin-top: 4px;
}

/* Glass panels */
.panel {
    position: relative;
    background:
        linear-gradient(135deg, rgba(15,23,42,0.86), rgba(30,41,59,0.68));
    border: 1px solid var(--border);
    border-radius: 26px;
    padding: 24px;
    box-shadow:
        0 22px 60px rgba(0,0,0,0.36),
        inset 0 1px 0 rgba(255,255,255,0.08);
    backdrop-filter: blur(20px);
}

.panel::before {
    content: "";
    position: absolute;
    inset: 0;
    border-radius: 26px;
    padding: 1px;
    background: linear-gradient(135deg, rgba(0,229,255,0.6), rgba(124,58,237,0.35), transparent);
    -webkit-mask:
        linear-gradient(#fff 0 0) content-box,
        linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
}

.panel-title {
    font-size: 19px;
    font-weight: 900;
    color: #f8fafc !important;
    margin-bottom: 7px;
}

.panel-desc {
    color: #94a3b8 !important;
    font-size: 14px;
    margin-bottom: 16px;
}

/* Input fields */
textarea,
input,
select {
    background: rgba(2, 6, 23, 0.68) !important;
    border: 1px solid rgba(148,163,184,0.28) !important;
    color: #e5e7eb !important;
    border-radius: 16px !important;
}

textarea:focus,
input:focus {
    border-color: rgba(0,229,255,0.75) !important;
    box-shadow: 0 0 0 3px rgba(0,229,255,0.12) !important;
}

label,
span,
p {
    color: #cbd5e1 !important;
}

/* File upload */
[data-testid="file"] {
    background: rgba(2, 6, 23, 0.62) !important;
    border: 1.5px dashed rgba(0,229,255,0.42) !important;
    border-radius: 20px !important;
    padding: 14px !important;
    transition: all 0.25s ease;
}

[data-testid="file"]:hover {
    border-color: rgba(0,229,255,0.9) !important;
    box-shadow: 0 0 24px rgba(0,229,255,0.16);
    transform: translateY(-2px);
}

/* Buttons */
button.primary {
    background: linear-gradient(135deg, #00e5ff, #2563eb, #7c3aed) !important;
    color: white !important;
    border: none !important;
    border-radius: 16px !important;
    font-weight: 900 !important;
    padding: 13px 18px !important;
    box-shadow:
        0 0 20px rgba(0,229,255,0.28),
        0 14px 36px rgba(37,99,235,0.35) !important;
    transition: all 0.25s ease !important;
}

button.primary:hover {
    transform: translateY(-3px) scale(1.01);
    box-shadow:
        0 0 32px rgba(0,229,255,0.45),
        0 20px 48px rgba(124,58,237,0.38) !important;
}

button.secondary,
button {
    border-radius: 16px !important;
    font-weight: 800 !important;
}

/* Dataframe */
.dataframe {
    border-radius: 22px !important;
    overflow: hidden !important;
    border: 1px solid rgba(0,229,255,0.22) !important;
    box-shadow: 0 18px 45px rgba(0,0,0,0.28);
}

table {
    background: rgba(15,23,42,0.85) !important;
}

thead tr th {
    background: linear-gradient(135deg, #0f172a, #1e3a8a) !important;
    color: #67e8f9 !important;
    font-weight: 900 !important;
    border-color: rgba(0,229,255,0.18) !important;
}

tbody tr td {
    background: rgba(15,23,42,0.72) !important;
    color: #e5e7eb !important;
    border-color: rgba(148,163,184,0.15) !important;
}

tbody tr:hover td {
    background: rgba(30,64,175,0.28) !important;
}

/* PDF download component */
[data-testid="file"] a,
[data-testid="file"] button {
    background: linear-gradient(135deg, #22c55e, #0ea5e9) !important;
    color: #ffffff !important;
    border-radius: 14px !important;
    font-weight: 900 !important;
    border: none !important;
}

/* Info strip */
#info-strip {
    margin-top: 24px;
    padding: 20px;
    border-radius: 24px;
    background:
        linear-gradient(135deg, rgba(15,23,42,0.82), rgba(30,41,59,0.62));
    border: 1px solid rgba(0,229,255,0.20);
    display: grid;
    grid-template-columns: repeat(5, 1fr);
    gap: 14px;
    box-shadow: 0 18px 45px rgba(0,0,0,0.28);
}

.info-item {
    text-align: center;
    padding: 16px 12px;
    border-radius: 18px;
    background: rgba(2, 6, 23, 0.55);
    border: 1px solid rgba(148,163,184,0.15);
    transition: all 0.25s ease;
}

.info-item:hover {
    transform: translateY(-3px);
    border-color: rgba(0,229,255,0.45);
    box-shadow: 0 0 22px rgba(0,229,255,0.14);
}

.info-title {
    font-weight: 900;
    color: #67e8f9 !important;
}

.info-sub {
    font-size: 12px;
    color: #94a3b8 !important;
}

.footer {
    text-align: center;
    color: #94a3b8 !important;
    font-size: 13px;
    margin-top: 20px;
}

/* Scrollbar */
::-webkit-scrollbar {
    width: 10px;
    height: 10px;
}

::-webkit-scrollbar-track {
    background: #020617;
}

::-webkit-scrollbar-thumb {
    background: linear-gradient(135deg, #00e5ff, #7c3aed);
    border-radius: 999px;
}

/* Mobile */
@media (max-width: 900px) {
    .metric-grid,
    #info-strip {
        grid-template-columns: 1fr 1fr;
    }

    #hero-title {
        font-size: 34px;
    }

    #hero {
        padding: 30px 24px;
    }
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="blue")) as demo:

    gr.HTML("""
    <div id="hero">
        <div id="badge">AI-Powered Recruitment Screening</div>
        <div id="hero-title"> Resume Screening & Ranking System</div>
        <div id="hero-subtitle">
            Upload multiple student resumes and compare them against a job description.
            The system ranks candidates using weighted skill matching, programming language relevance,
            semantic similarity, keyword similarity, and experience analysis.
        </div>

        <div class="metric-grid">
            <div class="metric-card">
                <div class="metric-value">35%</div>
                <div class="metric-label">Skill Match Weight</div>
            </div>
            <div class="metric-card">
                <div class="metric-value">25%</div>
                <div class="metric-label">Language Match Weight</div>
            </div>
            <div class="metric-card">
                <div class="metric-value">20%</div>
                <div class="metric-label">Semantic Similarity</div>
            </div>
            <div class="metric-card">
                <div class="metric-value">10% + 10%</div>
                <div class="metric-label">Keyword + Experience</div>
            </div>
        </div>
    </div>
    """)

    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            with gr.Group(elem_classes="panel"):
                gr.HTML("""
                <div class="panel-title">Candidate Resume Upload</div>
                <div class="panel-desc">
                    Upload PDF, DOCX, or TXT resumes. Multiple files are supported.
                </div>
                """)

                resume_files = gr.File(
                    label="Upload Student Resumes",
                    file_count="multiple",
                    file_types=[".pdf", ".docx", ".txt"]
                )

                gr.HTML("""
                <div class="panel-title" style="margin-top:18px;">Job Description Input</div>
                <div class="panel-desc">
                    Paste the target job description. Skills and languages will be extracted automatically.
                </div>
                """)

                job_description = gr.Textbox(
                    label="Job Description",
                    lines=15,
                    placeholder="Example: Looking for a Python Software Developer skilled in Java, HTML, CSS, C, C++, Python..."
                )

                with gr.Row():
                    submit_btn = gr.Button(
                        "Analyze & Rank Resumes",
                        variant="primary"
                    )
                    clear_btn = gr.ClearButton(
                        components=[],
                        value="Reset"
                    )

        with gr.Column(scale=2):
            with gr.Group(elem_classes="panel"):
                gr.HTML("""
                <div class="panel-title">Ranked Candidate Results</div>
                <div class="panel-desc">
                    Results are sorted by final score. Higher scores indicate stronger relevance to the job description.
                </div>
                """)

                output_table = gr.Dataframe(
                    label="Ranking Output",
                    wrap=True,
                    interactive=False
                )

    gr.HTML("""
    <div id="info-strip">
        <div class="info-item">
            <div class="info-title">Skills</div>
            <div class="info-sub">Matched & missing skills</div>
        </div>
        <div class="info-item">
            <div class="info-title">Languages</div>
            <div class="info-sub">Java, Python, C, C++</div>
        </div>
        <div class="info-item">
            <div class="info-title">Similarity</div>
            <div class="info-sub">Sentence-BERT + TF-IDF</div>
        </div>
        <div class="info-item">
            <div class="info-title">Experience</div>
            <div class="info-sub">Years-based scoring</div>
        </div>
        <div class="info-item">
            <div class="info-title">Ranking</div>
            <div class="info-sub">0–100% final score</div>
        </div>
    </div>

    <div class="footer">
        Built for explainable AI-based resume screening and student candidate ranking.
    </div>
    """)

    submit_btn.click(
        fn=rank_bulk_resumes,
        inputs=[resume_files, job_description],
        outputs=output_table
    )

demo.launch(share=True)

/tmp/ipykernel_937/4221295573.py:369: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="blue")) as demo:
/tmp/ipykernel_937/4221295573.py:369: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="blue")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://68d63c6f7f7c9035b2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
